# Regional Solar Energy Potential Analysis

This practical evaluates solar irradiance, compares fixed and ideal dual-axis tracking PV systems, and estimates rooftop electricity generation for a selected city.

## Prepare the environment

In Colab, select **File > Save a copy in Drive** and use a CPU runtime. Run the first cell to download the code and four-city data and install the required libraries. Continue in order. If setup requests a restart after updating a loaded library, restart the session and rerun this cell.

For local Jupyter, extract the complete package and open this Notebook from its folder. The same cell prepares the environment.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
import os
import runpy
import sys
import tempfile
import urllib.request
import zipfile

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    COURSE_URL = "https://raw.githubusercontent.com/Kechuan-ln/roof-pv-course-student/420fa692ddb3b68eb2fdddd56019ba9b63dbd05b/undergraduate/assets/course_runtime.zip"
    COURSE_SHA256 = "44653a69ea7db404bf7ea82a3b7630e2e7ec592b484e6568abfd8f01a3913edc"
    course_root = Path('/content') / ('pv_undergraduate_' + COURSE_SHA256[:12])
    if not (course_root / '.course_ready').is_file():
        if course_root.exists():
            raise RuntimeError('An incomplete course folder exists. Start a new Colab session and rerun this cell.')
        print('Downloading the course and four-city weather and building data...', flush=True)
        with tempfile.TemporaryDirectory(dir='/content') as temporary:
            folder = Path(temporary)
            archive = folder / 'course.zip'
            request = urllib.request.Request(COURSE_URL, headers={'User-Agent': 'PV-Course-Notebook'})
            with urllib.request.urlopen(request, timeout=120) as response, archive.open('wb') as output:
                import shutil
                shutil.copyfileobj(response, output)
            if hashlib.sha256(archive.read_bytes()).hexdigest() != COURSE_SHA256:
                raise RuntimeError('Course download failed its integrity check. Rerun this cell.')
            extracted = folder / 'course'
            extracted.mkdir()
            with zipfile.ZipFile(archive) as package:
                for entry in package.infolist():
                    target = (extracted / entry.filename).resolve()
                    if not target.is_relative_to(extracted.resolve()) or (entry.external_attr >> 16) & 0o170000 == 0o120000:
                        raise RuntimeError('Invalid course archive entry.')
                package.extractall(extracted)
            (extracted / '.course_ready').write_text(COURSE_SHA256)
            extracted.rename(course_root)
    if (course_root / '.course_ready').read_text() != COURSE_SHA256:
        raise RuntimeError('Course version mismatch. Start a new Colab session.')
    ROOT = runpy.run_path(str(course_root / 'scripts/prepare_environment.py'))['prepare'](course_root)
else:
    candidates = [candidate for parent in [Path.cwd(), *Path.cwd().parents]
                  for candidate in [parent, parent / 'student']]
    ROOT = next((p for p in candidates if (p / 'pv_course').is_dir()
                 and (p / 'scripts/prepare_environment.py').is_file()), None)
    if ROOT is None:
        raise RuntimeError('Extract the full student package and open Solar_PV_Practical.ipynb from that folder.')
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    runpy.run_path(str(ROOT / 'scripts/prepare_environment.py'))['prepare'](ROOT)

## 1 Solar irradiance

Select Bangkok, Shenzhen, Seoul or Tokyo. The weather data contain 10-minute values of global horizontal irradiance (GHI), direct normal irradiance (DNI) and diffuse horizontal irradiance (DHI), in W/m². Year and analysis date are read from the selected dataset.

The figures show a selected day, the mean daily profile, monthly daily profiles, monthly means and seasonal means. Means include nighttime values. Seasonal groups use DJF, MAM, JJA and SON.

Compare the timing and relative magnitudes of GHI, DNI and DHI. Identify the months with the highest mean GHI.

In [ ]:
%matplotlib inline
import json
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import display
from pv_course import weather_visualization as weather_plots
from pv_course import poa_visualization as poa_plots
from pv_course.poa_calculation import get_timezone, calculate_poa, calculate_poa_track
from pv_course.data_quality import interval_energy, coverage_report
from pv_course.notebook_io import show_figure, download_results

CITY = 'Shenzhen'  # Bangkok, Shenzhen, Seoul or Tokyo
DAY = None  # None selects the first day; alternatively use YYYY-MM-DD.
MODULE_AREA = 1.69  # m² per module
ROOF_FRACTION = 0.80
STEP_MINUTES = 10

weather = weather_plots.read_weather(ROOT / 'weather_data' / f'{CITY}_weather.csv')
footprints = gpd.read_file(ROOT / 'buildings_data' / f'{CITY}_building.geojson')
centroids = footprints.to_crs(footprints.estimate_utm_crs()).centroid.to_crs(4326)
latitude, longitude = float(centroids.y.mean()), float(centroids.x.mean())
weather.index = weather.index.tz_localize(get_timezone(latitude, longitude))
YEAR = int(weather.index[0].year)
DAY = weather.index[0].strftime('%Y-%m-%d') if DAY is None else DAY
OUTPUT = ROOT / 'result' / CITY
FIGURES = OUTPUT / 'figures'
OUTPUT.mkdir(parents=True, exist_ok=True)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})
print(f'{CITY}: {len(weather):,} records, {weather.index[0]} to {weather.index[-1]}')
print(f'Location: {latitude:.4f}°, {longitude:.4f}°; analysis date: {DAY}')
coverage = coverage_report(weather, STEP_MINUTES)
display(pd.Series({k: coverage[k] for k in ['records', 'calendar_year', 'missing_calendar_intervals', 'calendar_coverage']}))

weather_plots.plot_single_day(weather, DAY)
show_figure('01_daily_irradiance', FIGURES)
for function, name in [(weather_plots.plot_daily_avg, '02_mean_daily_profile'),
                       (weather_plots.plot_monthly_avg, '03_monthly_daily_profiles'),
                       (weather_plots.plot_monthly_hourly, '04_monthly_irradiance'),
                       (weather_plots.plot_seasonal, '05_seasonal_irradiance')]:
    function(weather)
    show_figure(name, FIGURES)

## 2 Plane of array irradiance

Plane of array (POA) irradiance is the irradiance incident on the module plane. It comprises direct, sky-diffuse and ground-reflected components.

The fixed system uses a tilt equal to the absolute latitude and an equator-facing azimuth. Ideal dual-axis tracking follows the Sun. The calculation uses the Perez transposition model and a ground albedo of 0.2.

### Fixed system

In [ ]:
SURFACE_TILT = abs(latitude)  # Degrees above horizontal; change to investigate tilt.
SURFACE_AZIMUTH = 180.0 if latitude >= 0 else 0.0
fixed = pd.DataFrame(calculate_poa(weather.copy(), SURFACE_TILT, SURFACE_AZIMUTH, latitude, longitude))
tracking = pd.DataFrame(calculate_poa_track(weather.copy(), latitude, longitude))
fixed.to_csv(OUTPUT / 'poa_fixed.csv', index_label='timestamp')
tracking.to_csv(OUTPUT / 'poa_tracking.csv', index_label='timestamp')
poa_plots.plot_poa_single_day(fixed, DAY)
show_figure('06_fixed_poa_day', FIGURES)
for function, name in [(poa_plots.plot_poa_daily_avg, '07_fixed_poa_daily_profile'),
                       (poa_plots.plot_poa_monthly_avg, '08_fixed_poa_monthly'),
                       (poa_plots.plot_poa_seasonal, '09_fixed_poa_seasonal')]:
    function(fixed)
    show_figure(name, FIGURES)

### Tracking system

Compare the direct component with the fixed system at the same times. Examine whether the tracking advantage changes across months.

In [ ]:
poa_plots.plot_poa_single_day(tracking, DAY)
show_figure('10_tracking_poa_day', FIGURES)
for function, name in [(poa_plots.plot_poa_daily_avg, '11_tracking_poa_daily_profile'),
                       (poa_plots.plot_poa_monthly_avg, '12_tracking_poa_monthly'),
                       (poa_plots.plot_poa_seasonal, '13_tracking_poa_seasonal')]:
    function(tracking)
    show_figure(name, FIGURES)

### Fixed and tracking comparison

Mean irradiance is measured in W/m². Time-integrated irradiance, or irradiation, is measured in kWh/m². Each record is treated as a 10-minute mean:

$$H=\sum_t G_t\Delta t/1000,\qquad \Delta t=10/60\;\mathrm{h}.$$

In [ ]:
poa_plots.compare_fixed_tracked_daily(fixed, tracking)
show_figure('14_poa_comparison', FIGURES)
poa_totals = pd.Series({
    'Fixed POA irradiation (kWh/m²)': interval_energy(fixed.total_poa, STEP_MINUTES).sum(min_count=1) / 1000,
    'Tracking POA irradiation (kWh/m²)': interval_energy(tracking.total_poa, STEP_MINUTES).sum(min_count=1) / 1000,
})
display(poa_totals.to_frame('Value'))

## 3 PV module output

The PVsyst single-diode model calculates the DC power of one module. The SAPM temperature model estimates cell temperature from POA irradiance, air temperature and wind speed. The scenario represents clean, unshaded modules before wiring and inverter losses.

The electrical parameters below define an example module. Retain them when comparing the two mounting systems.

In [ ]:
from pv_course.Efficiency_calculation import calculate_pv_efficiency

MODULE_PARAMETERS = {
    'gamma_ref': 1.0,       # Diode ideality factor at reference conditions
    'mu_gamma': 0.002,      # Temperature coefficient of diode ideality factor, 1/K
    'I_L_ref': 7.3,         # Reference light-generated current, A
    'I_o_ref': 1e-10,       # Reference reverse saturation current, A
    'R_sh_ref': 400.0,      # Reference shunt resistance, ohm
    'R_s': 0.3,             # Series resistance, ohm
    'R_sh_0': 3000.0,       # Shunt resistance at zero irradiance, ohm
    'cells_in_series': 60,
    'alpha_sc': 0.004,      # Short-circuit current temperature coefficient, A/K
    'EgRef': 1.12,          # Reference bandgap energy, eV
    'irrad_ref': 1000.0,    # Reference irradiance, W/m²
    'temp_ref': 25.0,       # Reference cell temperature, °C
}
power_fixed = calculate_pv_efficiency(fixed, weather, MODULE_PARAMETERS, MODULE_AREA)
power_tracking = calculate_pv_efficiency(tracking, weather, MODULE_PARAMETERS, MODULE_AREA)
for name, data in [('fixed', power_fixed), ('tracking', power_tracking)]:
    data.drop(columns='timestamp').to_csv(OUTPUT / f'power_{name}.csv', index_label='timestamp')
poa_plots.plot_pv_power_daily(power_fixed, power_tracking)
show_figure('15_mean_daily_power', FIGURES)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(power_fixed.loc[DAY].index, power_fixed.loc[DAY, 'power'], label='Fixed')
ax.plot(power_tracking.loc[DAY].index, power_tracking.loc[DAY, 'power'], label='Tracking')
ax.set(ylabel='Module DC power (W)', xlabel='Local time', title=f'Module output on {DAY}')
from matplotlib.dates import DateFormatter
ax.xaxis.set_major_formatter(DateFormatter('%H:%M', tz=weather.index.tz))
ax.legend(); fig.autofmt_xdate()
show_figure('16_module_power_day', FIGURES)

### Power and energy

Power is measured in W; energy is measured in kWh. Integrate the module power over time:

$$E=\sum_t P_t\Delta t/1000.$$

Energy totals cover the supplied records. Shenzhen contains 2020 data with February 29 missing; the other cities contain complete 2019 timestamp records. Compare monthly energy and quantify the tracking gain.

In [ ]:
energy = pd.DataFrame({
    'Fixed': interval_energy(power_fixed.power, STEP_MINUTES) / 1000,
    'Tracking': interval_energy(power_tracking.power, STEP_MINUTES) / 1000,
})
monthly_energy = energy.resample('MS').sum(min_count=1)
monthly_energy.to_csv(OUTPUT / 'monthly_module_energy.csv', index_label='timestamp')
monthly_energy.set_axis(monthly_energy.index.strftime('%b')).plot.bar(figsize=(12, 4), rot=0)
plt.ylabel('Module DC energy (kWh)'); plt.xlabel('Month')
plt.title('Monthly module energy')
show_figure('17_monthly_energy', FIGURES)
module_totals = energy.sum(min_count=1)
summary = pd.DataFrame({'Module energy (kWh)': module_totals,
                        'Yield per module area (kWh/m²)': module_totals / MODULE_AREA})
display(summary)
print(f'Tracking gain: {(module_totals.Tracking / module_totals.Fixed - 1) * 100:.1f}%')
summary.to_csv(OUTPUT / 'module_summary.csv', index_label='system')

## 4 Rooftop PV potential

Roof areas are calculated in a local UTM coordinate system. With a usable roof fraction $f$, building energy is

$$E_\mathrm{building}=\frac{E_\mathrm{module}}{A_\mathrm{module}}A_\mathrm{roof}f.$$

All buildings share the same weather and installation assumptions, so estimated energy is proportional to usable roof area. The fixed and tracking scenarios use the same installed module area. The footprint audit reports exact duplicates and overlapping areas; the table shows both summed-footprint and union-area scenarios for the region.

In [ ]:
from pv_course.building_pv_potential import calculate_pv_potential, create_statistics_plot

buildings = calculate_pv_potential(footprints, power_fixed, roof_efficiency=ROOF_FRACTION,
                                  module_area=MODULE_AREA, step_minutes=STEP_MINUTES)
audit = buildings.attrs['geometry_audit']
local_buildings = buildings.to_crs(buildings.estimate_utm_crs())
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
local_buildings.plot(ax=axes[0], facecolor='#c6dbe6', edgecolor='#4b6272', linewidth=0.2)
axes[0].set_title('Building footprints')
local_buildings.plot(column='energy_kwh', ax=axes[1], legend=True, cmap='YlOrRd',
                     legend_kwds={'label': 'DC energy over supplied records (kWh)', 'shrink': 0.7})
axes[1].set_title('Fixed-system rooftop energy')
for ax in axes:
    ax.set_xlabel('Easting (m)'); ax.set_ylabel('Northing (m)')
plt.tight_layout()
show_figure('18_rooftop_maps', FIGURES)
create_statistics_plot(buildings)
show_figure('19_rooftop_statistics', FIGURES)
yield_per_area = module_totals / MODULE_AREA
region = pd.DataFrame({
    'Summed-footprint energy (MWh)': yield_per_area * audit['sum_area_m2'] * ROOF_FRACTION / 1000,
    'Union-area energy (MWh)': yield_per_area * audit['union_area_m2'] * ROOF_FRACTION / 1000,
})
display(region)
print(f'Buildings retained: {len(buildings):,}; exact duplicates removed: {audit["exact_duplicates_removed"]}')
print(f'Summed roof area: {audit["sum_area_m2"]:,.0f} m²; overlap fraction: {audit["overlap_fraction"]:.2%}')
region.to_csv(OUTPUT / 'regional_energy_scenarios.csv', index_label='system')
buildings.drop(columns='geometry').to_csv(OUTPUT / 'building_energy.csv', index=False)

### Selected building

Select a value from the `building_id` column. The default selects the largest estimated energy output. Compare its daily and monthly energy under the two mounting scenarios. Explain how its estimate changes if the usable roof fraction decreases from 0.8 to 0.5.

In [ ]:
BUILDING_ID = None
if BUILDING_ID is None:
    BUILDING_ID = buildings.loc[buildings.energy_kwh.idxmax(), 'building_id']
selected = buildings.loc[buildings.building_id.astype(str) == str(BUILDING_ID)]
if selected.empty:
    raise ValueError('Choose a building_id from the building table.')
building = selected.iloc[0]
factor = building.usable_area / MODULE_AREA
building_daily = energy.resample('D').sum(min_count=1) * factor
building_monthly = monthly_energy * factor
building_daily.plot(figsize=(12, 4))
plt.ylabel('Building DC energy (kWh/day)'); plt.xlabel('Date')
plt.title(f'Daily energy for building {BUILDING_ID}')
show_figure('20_building_daily_energy', FIGURES)
building_monthly.set_axis(building_monthly.index.strftime('%b')).plot.bar(figsize=(12, 4), rot=0)
plt.ylabel('Building DC energy (kWh/month)'); plt.xlabel('Month')
plt.title(f'Monthly energy for building {BUILDING_ID}')
show_figure('21_building_monthly_energy', FIGURES)
display(pd.Series({'Building ID': BUILDING_ID, 'Roof area (m²)': building.roof_area,
                   'Usable area (m²)': building.usable_area,
                   'Fixed energy (kWh)': module_totals.Fixed * factor,
                   'Tracking energy (kWh)': module_totals.Tracking * factor}))
building_daily.to_csv(OUTPUT / 'selected_building_daily.csv', index_label='timestamp')
building_monthly.to_csv(OUTPUT / 'selected_building_monthly.csv', index_label='timestamp')

## Summary and export

Summarize the city's solar resource, the fixed-versus-tracking comparison and one building's estimated energy. Support each conclusion with a figure and a numerical result. State the data period, tilt, module area and usable roof fraction.

The final cell downloads figures, CSV tables and calculation settings. Save your edited Notebook separately using **File > Download > Download .ipynb** in Colab.

In [ ]:
settings = {'city': CITY, 'year': YEAR, 'analysis_date': DAY,
            'latitude': latitude, 'longitude': longitude, 'timezone': str(weather.index.tz),
            'surface_tilt': SURFACE_TILT, 'surface_azimuth': SURFACE_AZIMUTH,
            'module_area_m2': MODULE_AREA, 'usable_roof_fraction': ROOF_FRACTION,
            'step_minutes': STEP_MINUTES, 'selected_building_id': str(BUILDING_ID),
            'module_parameters': MODULE_PARAMETERS, 'coverage': coverage,
            'geometry_audit': audit}
(OUTPUT / 'settings.json').write_text(json.dumps(settings, indent=2))
results_zip = download_results(ROOT)